# KG1 v73 — GRPO TRL Stage-2 (Colab Pro+ A100)

## Framework: priyanlc/autoresearch-sft-grpo + dtdo90 rewards

**Bombas**:
- 6 rewards compostos (priyanlc): correctness + format + reasoning + category_bonus + single_box + final_line
- KL beta=0.01, num_generations=4, LR=5e-6
- Foco: bit_manip 3-input + cryptarithm + equation_guess (gaps huikang)
- Memory: ~36-38GB A100 (vllm rollouts otimizados)

## Pré-requisito: V73 SFT adapter (FASE 2 completa) em Drive ou HF

## Score esperado: 0.86 → 0.87 (P=70%+)

In [ ]:
# Cell 1: Setup
import torch, subprocess, os
r = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
print(r.stdout[:800])

from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()};setInterval(ClickConnect, 60000)'))

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q --no-deps 'trl>=0.16' peft>=0.18.1 accelerate bitsandbytes vllm>=0.6
%pip install -q 'transformers>=4.55' datasets liger-kernel

from google.colab import drive, userdata
drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_KEY')
os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
# Cell 2: Load V73 SFT adapter (FASE 2) como ponto de partida
from unsloth import FastLanguageModel

MAX_SEQ = 4096
model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)

# Carregar V73 SFT adapter como base
from peft import PeftModel
V73_SFT_PATH = '/content/drive/MyDrive/kg1_v73_unsloth_moe/final_adapter'
if not os.path.exists(V73_SFT_PATH):
    # Fallback: download from HF
    from huggingface_hub import snapshot_download
    V73_SFT_PATH = snapshot_download(
        'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe',
        token=HF_TOKEN, allow_patterns=['final/*']
    ) + '/final'

model = PeftModel.from_pretrained(model, V73_SFT_PATH, adapter_name='sft', is_trainable=True)
model.set_adapter('sft')
print(f'Loaded V73 SFT from {V73_SFT_PATH}')

In [ ]:
# Cell 3: Reward functions (priyanlc + dtdo90 compositos)
import re
from src.competition_utils import extract_final_answer, verify  # nosso util

BOXED_RE = re.compile(r'\\boxed\{([^{}]+)\}')
FINAL_LINE_RE = re.compile(r'(?:Answer|Final|Result|Resposta)[:\s]+([^\n]+)', re.I)

W_CORRECTNESS = 1.0
W_FORMAT = 0.3
W_REASONING = 0.15
W_CATEGORY_BONUS = 0.2
W_SINGLE_BOX = 0.08
W_FINAL_LINE = 0.02

def reward_correctness(prompts, completions, answers, **kwargs):
    rewards = []
    for c, a in zip(completions, answers):
        pred = extract_final_answer(c)
        rewards.append(W_CORRECTNESS if verify(str(a), str(pred)) else 0.0)
    return rewards

def reward_format(prompts, completions, **kwargs):
    return [W_FORMAT if BOXED_RE.search(c) else 0.0 for c in completions]

def reward_single_box(prompts, completions, **kwargs):
    return [W_SINGLE_BOX if len(BOXED_RE.findall(c)) == 1 else 0.0 for c in completions]

def reward_final_line(prompts, completions, **kwargs):
    return [W_FINAL_LINE if FINAL_LINE_RE.search(c) else 0.0 for c in completions]

def reward_reasoning(prompts, completions, **kwargs):
    # Reward reasoning length: 500-3000 chars sweet spot
    rewards = []
    for c in completions:
        L = len(c)
        if 500 <= L <= 3000:
            rewards.append(W_REASONING)
        elif L > 3000:
            rewards.append(W_REASONING * 0.5)
        else:
            rewards.append(0.0)
    return rewards

def reward_category_bonus(prompts, completions, families, **kwargs):
    # Bonus se output contem markers da categoria correta
    markers = {
        'bit_manipulation': ['XOR', 'AND', 'OR', 'NOT', 'binary'],
        'cipher': ['substitution', 'mapping', 'decrypt'],
        'equation_numeric': ['operator', 'arithmetic'],
        'gravity': ['g =', 'd =', 't ='],
        'unit_conversion': ['multiply', 'convert'],
    }
    rewards = []
    for c, fam in zip(completions, families):
        ms = markers.get(fam, [])
        hits = sum(1 for m in ms if m.lower() in c.lower())
        rewards.append(W_CATEGORY_BONUS * min(1.0, hits / max(1, len(ms))))
    return rewards

REWARD_FUNCS = [reward_correctness, reward_format, reward_single_box, 
                reward_final_line, reward_reasoning, reward_category_bonus]
print(f'Loaded {len(REWARD_FUNCS)} reward functions')

In [ ]:
# Cell 4: Dataset GRPO - foco em hard examples (bit_manip 3-input + cryptarithm + equation_guess)
from datasets import load_dataset

ds_full = load_dataset('felipesp1983/kg1-nemotron-training', 
                       data_files='data/sft_v70_huikang_full.jsonl',
                       split='train', token=HF_TOKEN)

# Filter hard families only
HARD_FAMILIES = ['bit_manipulation', 'cryptarithm_deduce', 'cryptarithm_guess', 
                  'equation_numeric_guess', 'equation_numeric_deduce']
ds_hard = ds_full.filter(
    lambda x: x.get('family', x.get('category', '')) in HARD_FAMILIES,
    num_proc=4
)
print(f'Hard examples: {len(ds_hard)} (de {len(ds_full)})')

# Limitar para 600 prompts (priyanlc usou 120, escalando 5x)
ds_grpo = ds_hard.shuffle(seed=42).select(range(min(600, len(ds_hard))))

def to_grpo(ex):
    msgs = ex['messages']
    user_msg = next((m['content'] for m in msgs if m['role'] == 'user'), '')
    assistant_msg = next((m['content'] for m in reversed(msgs) if m['role'] == 'assistant'), '')
    answer_match = BOXED_RE.search(assistant_msg)
    answer = answer_match.group(1) if answer_match else ''
    return {'prompt': user_msg, 'answer': answer, 'family': ex.get('family', ex.get('category', 'unknown'))}

ds_grpo = ds_grpo.map(to_grpo, num_proc=4, remove_columns=ds_grpo.column_names)
print(f'GRPO ready: {len(ds_grpo)} prompts')

In [ ]:
# Cell 5: GRPO Training
from trl import GRPOTrainer, GRPOConfig

CKPT_DIR = '/content/drive/MyDrive/kg1_v73_grpo'
os.makedirs(CKPT_DIR, exist_ok=True)

grpo_args = GRPOConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,                    # rollouts (memory = 4x)
    max_prompt_length=2048,
    max_completion_length=4096,
    learning_rate=5e-6,                   # priyanlc proven
    beta=0.01,                            # KL low
    num_train_epochs=1,                   # 600 prompts × 1 epoch ~ 300 steps eff
    save_steps=50,
    logging_steps=5,
    bf16=True,
    report_to='none',
    push_to_hub=False,
    seed=42,
    use_vllm=True,                        # vLLM rollouts (-15GB)
    vllm_gpu_memory_utilization=0.3,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=REWARD_FUNCS,
    args=grpo_args,
    train_dataset=ds_grpo,
)

trainer.train(resume_from_checkpoint=os.path.exists(f'{CKPT_DIR}/checkpoint-50'))
trainer.save_model(f'{CKPT_DIR}/final_grpo')
print('GRPO done')

In [ ]:
# Cell 6: Upload V73-GRPO adapter
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-grpo'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=f'{CKPT_DIR}/final_grpo', repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded V73-GRPO to {REPO_ID}')